# Yangjiang station wind, VRMSE, terrain, and LCZ analysis

## tl;dr

This notebook evaluates both 2-min-mean and 10-min-mean station wind observations against five AI-WRF simulations over the complete Yangjiang window from 2025-09-23 09:00 to 2025-09-25 09:00 UTC. It produces four all-station time-series figures, station-wise vector RMSE (VRMSE) statistics and separate 2-min/10-min boxplots, a station-location map, and Yangjiang terrain and land-use/LCZ maps.

Across the 32 stations, Pangu-WRF has the lowest median and mean VRMSE for both averaging intervals: 7.427 and 8.136 m s⁻¹ for 2-min, and 7.234 and 8.057 m s⁻¹ for 10-min. The 10-min median and mean are slightly lower than the corresponding 2-min values for every driver. Missing observations and invalid wind directions are excluded pairwise, and every averaging-interval/model/station VRMSE retains its valid-sample count.


## Context & Methods

### Key assumptions

- “2-min” and “10-min” describe the averaging interval of each wind observation; both datasets contain hourly records.
- The complete common period is used: 49 hourly timestamps from 2025-09-23 09:00 to 2025-09-25 09:00 UTC, inclusive.
- The 2-min and 10-min results are calculated independently from their matching observation and simulation files.
- Only Pangu-WRF, GraphCast-WRF, FengWu-WRF, FuXi-WRF, and Aurora-WRF are compared; ERA5 is intentionally excluded.
- Meteorological wind direction is converted using $u=-S\sin\theta$ and $v=-S\cos\theta$.
- Station VRMSE is $\sqrt{\mathrm{mean}[(u_s-u_o)^2+(v_s-v_o)^2]}$ over common valid timestamps.
- No VRMSE spatial map is produced. Spatial context is provided by the station-location, terrain, and LCZ maps.


In [1]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm, LightSource, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import font_manager
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import netCDF4 as nc
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

def find_supplementary_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd / "Supplementary", cwd, *[p / "Supplementary" for p in cwd.parents]]
    for candidate in candidates:
        if (candidate / "yangjiang" / "stations").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate Supplementary/yangjiang from {cwd}")

SUPP_DIR = find_supplementary_dir()
DATA_DIR = SUPP_DIR / "yangjiang"
PUBLISH_DIR = SUPP_DIR / "publish"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

wind_candidates = [DATA_DIR / "wind_results", DATA_DIR / "station_wind"]
WIND_DIR = next((p for p in wind_candidates if p.is_dir()), None)
if WIND_DIR is None:
    raise FileNotFoundError(f"Neither wind_results nor station_wind exists under {DATA_DIR}")

OBS_DIR = WIND_DIR / "results_obs_selected"
SIM_DIR = WIND_DIR / "results_sim_selected"
STATION_FILE = next((DATA_DIR / "stations").glob("*.xlsx"))
MAP_FILE = DATA_DIR / "terrain_lcz" / "maps_data_d05_yangjiang.nc"

MODELS = ["pangu", "graphcast", "fengwu", "fuxi", "aurora"]
MODEL_STYLES = {
    "pangu": {"label": "Pangu-WRF", "color": "#F58518", "marker": "s"},
    "graphcast": {"label": "GraphCast-WRF", "color": "#54A24B", "marker": "^"},
    "fengwu": {"label": "FengWu-WRF", "color": "#E45756", "marker": "D"},
    "fuxi": {"label": "FuXi-WRF", "color": "#72B7B2", "marker": "v"},
    "aurora": {"label": "Aurora-WRF", "color": "#B279A2", "marker": "p"},
}
OBS_STYLE = {"label": "Observation", "color": "#2F2F2F", "marker": "o"}
SOURCE_ORDER = ["obs", *MODELS]
TS_COL = "timestamp_utc"
TS_FORMAT = "%Y%m%d %H%M"
DPI = 600

START_TIME = pd.Timestamp("2025-09-23 09:00", tz="UTC")
END_TIME = pd.Timestamp("2025-09-25 09:00", tz="UTC")
EXPECTED_HOURLY_RECORDS = int((END_TIME - START_TIME) / pd.Timedelta(hours=1)) + 1

AVERAGING_INTERVALS = ["2min", "10min"]
AVERAGING_STYLES = {
    "2min": {"label": "2-min", "obs_token": "2mi", "color": "#6E8BC3"},
    "10min": {"label": "10-min", "obs_token": "10mi", "color": "#C96A5A"},
}

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT_FAMILY = "Arial" if "Arial" in available_fonts else "DejaVu Sans"
mpl.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 12,
    "savefig.facecolor": "white",
})

print(f"Supplementary directory: {SUPP_DIR}")
print(f"Wind directory: {WIND_DIR}")
print("Station metadata: " + STATION_FILE.name.encode("ascii", "backslashreplace").decode())
print(f"Map data: {MAP_FILE.name}")
print(f"Output directory: {PUBLISH_DIR}")


Supplementary directory: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary
Wind directory: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\yangjiang\wind_results
Station metadata: \u53f0\u98ce\u6866\u52a0\u6c99-\u9633\u6c5f\u7ad9\u70b9\u7b5b\u9009-\u7ad9\u70b9\u4fe1\u606f.xlsx
Map data: maps_data_d05_yangjiang.nc
Output directory: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish


## Data

### 1. Load and validate station metadata and wind data

Station identifiers are normalized to lower case internally. Blank values, negative wind speeds, and wind directions outside 0–360° are treated as missing. The validation cell checks time ordering, hourly continuity, station membership, and input alignment before any calculation.


In [2]:
def read_station_metadata(path: Path) -> pd.DataFrame:
    station_meta = pd.read_excel(path, usecols=[0, 1, 2, 3, 4])
    station_meta.columns = ["station_name", "station", "lat", "lon", "elevation"]
    station_meta["station"] = station_meta["station"].astype(str).str.strip().str.lower()
    station_meta[["lat", "lon", "elevation"]] = station_meta[["lat", "lon", "elevation"]].apply(
        pd.to_numeric, errors="coerce"
    )
    station_meta = station_meta.dropna(subset=["station", "lat", "lon"]).reset_index(drop=True)
    if station_meta["station"].duplicated().any():
        raise ValueError("Duplicate stations in station metadata")
    return station_meta


def read_wind_csv(path: Path, *, direction: bool = False) -> tuple[pd.DataFrame, dict[str, int]]:
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, dtype={0: str})
    frame = frame.rename(columns={frame.columns[0]: TS_COL})
    frame = frame.rename(columns={c: str(c).strip().lower() for c in frame.columns if c != TS_COL})
    frame[TS_COL] = pd.to_datetime(
        frame[TS_COL].astype(str).str.strip(), format=TS_FORMAT, utc=True, errors="raise"
    )
    if frame[TS_COL].duplicated().any():
        raise ValueError(f"Duplicate timestamps in {path}")
    frame = frame.sort_values(TS_COL).set_index(TS_COL)
    frame = frame.apply(pd.to_numeric, errors="coerce")
    non_numeric_or_blank = int(frame.isna().sum().sum())
    invalid = (frame.lt(0) | frame.gt(360)) if direction else frame.lt(0)
    invalid_count = int(invalid.sum().sum())
    frame = frame.mask(invalid)
    return frame, {"blank_or_non_numeric": non_numeric_or_blank, "out_of_range": invalid_count}


station_meta = read_station_metadata(STATION_FILE)
station_order = station_meta["station"].tolist()
expected_stations = set(station_order)

obs_speed_by_interval: dict[str, pd.DataFrame] = {}
obs_direction_by_interval: dict[str, pd.DataFrame] = {}
simulation_speed_by_interval: dict[str, dict[str, pd.DataFrame]] = {}
simulation_direction_by_interval: dict[str, dict[str, pd.DataFrame]] = {}

for averaging in AVERAGING_INTERVALS:
    obs_token = AVERAGING_STYLES[averaging]["obs_token"]
    obs_speed_by_interval[averaging], _ = read_wind_csv(
        OBS_DIR / f"aws_WIN_S_Avg_{obs_token}_yangjiang_selected.csv", direction=False
    )
    obs_direction_by_interval[averaging], _ = read_wind_csv(
        OBS_DIR / f"aws_WIN_D_Avg_{obs_token}_yangjiang_selected.csv", direction=True
    )
    simulation_speed_by_interval[averaging] = {}
    simulation_direction_by_interval[averaging] = {}
    for model in MODELS:
        simulation_speed_by_interval[averaging][model], _ = read_wind_csv(
            SIM_DIR / f"{model}_{averaging}_mean_wspd_sim_24h_selected.csv", direction=False
        )
        simulation_direction_by_interval[averaging][model], _ = read_wind_csv(
            SIM_DIR / f"{model}_{averaging}_mean_wdir_sim_24h_selected.csv", direction=True
        )

reference_index = obs_speed_by_interval["2min"].index
expected_full_index = pd.date_range(reference_index.min(), reference_index.max(), freq="1h", tz="UTC")
if not reference_index.equals(expected_full_index):
    raise ValueError("Wind data are not a complete hourly sequence")

all_frames = []
for averaging in AVERAGING_INTERVALS:
    all_frames.extend([
        obs_speed_by_interval[averaging],
        obs_direction_by_interval[averaging],
        *simulation_speed_by_interval[averaging].values(),
        *simulation_direction_by_interval[averaging].values(),
    ])
for frame in all_frames:
    if set(frame.columns) != expected_stations:
        raise ValueError("Station columns do not match the station metadata")
    if not frame.index.equals(reference_index):
        raise ValueError("All 2-min/10-min observation and simulation timestamps must align")

if START_TIME < reference_index.min() or END_TIME > reference_index.max():
    raise ValueError("Requested analysis window is outside the available wind data")
window_index = pd.date_range(START_TIME, END_TIME, freq="1h", tz="UTC")
if len(window_index) != EXPECTED_HOURLY_RECORDS:
    raise RuntimeError("Unexpected number of hourly records in the analysis window")

for averaging in AVERAGING_INTERVALS:
    obs_speed_by_interval[averaging] = obs_speed_by_interval[averaging].loc[window_index, station_order]
    obs_direction_by_interval[averaging] = obs_direction_by_interval[averaging].loc[window_index, station_order]
    simulation_speed_by_interval[averaging] = {
        model: frame.loc[window_index, station_order]
        for model, frame in simulation_speed_by_interval[averaging].items()
    }
    simulation_direction_by_interval[averaging] = {
        model: frame.loc[window_index, station_order]
        for model, frame in simulation_direction_by_interval[averaging].items()
    }

WINDOW_TAG = f"{START_TIME:%Y%m%d-%H}_to_{END_TIME:%Y%m%d-%H}"
quality_rows = []
for averaging in AVERAGING_INTERVALS:
    valid_pairs = (
        obs_speed_by_interval[averaging].notna()
        & obs_direction_by_interval[averaging].notna()
    ).sum(axis=0)
    quality_rows.append({
        "averaging": AVERAGING_STYLES[averaging]["label"],
        "speed_missing_or_invalid": int(obs_speed_by_interval[averaging].isna().sum().sum()),
        "direction_missing_or_invalid": int(obs_direction_by_interval[averaging].isna().sum().sum()),
        "min_valid_pairs_per_station": int(valid_pairs.min()),
        "max_valid_pairs_per_station": int(valid_pairs.max()),
    })

print(f"Stations: {len(station_order)}")
print(f"Hourly timestamps: {len(window_index)} ({START_TIME} to {END_TIME}, inclusive)")
print("Observation quality in the selected window:")
display(pd.DataFrame(quality_rows).set_index("averaging"))


Stations: 32
Hourly timestamps: 49 (2025-09-23 09:00:00+00:00 to 2025-09-25 09:00:00+00:00, inclusive)
Observation quality in the selected window:
           speed_missing_or_invalid  ...  max_valid_pairs_per_station
averaging                            ...                             
2-min                            31  ...                           49
10-min                           31  ...                           49

[2 rows x 4 columns]


## Results

### 2. All-station 2-min-mean wind time series

The speed panels use lines because continuous evolution is the comparison of interest. Direction uses points to avoid artificial lines across the 0°/360° boundary. Every panel uses the same axes and source styling.


In [3]:
def save_tiff(fig: mpl.figure.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    kwargs = dict(dpi=DPI, format="tiff", bbox_inches="tight")
    try:
        fig.savefig(path, pil_kwargs={"compression": "tiff_lzw"}, **kwargs)
    except TypeError:
        fig.savefig(path, **kwargs)
    return path


def source_style(source: str) -> dict:
    return OBS_STYLE if source == "obs" else MODEL_STYLES[source]


def time_ticks(start: pd.Timestamp, end: pd.Timestamp) -> tuple[list[float], list[str]]:
    times = pd.date_range(start, end, freq="6h", tz="UTC")
    ticks = mdates.date2num(times.to_pydatetime())
    labels = []
    previous_date = None
    for stamp in times.to_pydatetime():
        labels.append(stamp.strftime("%HZ\n%m-%d") if stamp.date() != previous_date else stamp.strftime("%HZ"))
        previous_date = stamp.date()
    return list(ticks), labels


def all_station_sources(observed: pd.DataFrame, simulations: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    return {"obs": observed, **{model: simulations[model] for model in MODELS}}


def plot_all_station_time_series(
    sources: dict[str, pd.DataFrame], metric: str, averaging: str, output_name: str
) -> Path:
    ncols = 5
    nrows = math.ceil(len(station_order) / ncols)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(16.5, 2.18 * nrows + 1.6), sharex=True, sharey=True
    )
    axes_flat = axes.ravel().tolist()

    finite_max = max(float(np.nanmax(frame.to_numpy())) for frame in sources.values())
    ymax = math.ceil(finite_max / 5.0) * 5.0 if metric == "speed" else 360.0

    for i, station in enumerate(station_order):
        ax = axes_flat[i]
        for source in SOURCE_ORDER:
            style = source_style(source)
            series = sources[source][station].dropna()
            if metric == "speed":
                ax.plot(
                    series.index, series.values, color=style["color"],
                    linewidth=0.85 if source == "obs" else 0.75,
                    alpha=0.85 if source == "obs" else 0.72,
                    marker=style["marker"], markevery=3, markersize=2.5,
                    markeredgecolor="black", markeredgewidth=0.25,
                    zorder=4 if source == "obs" else 2,
                )
            else:
                ax.scatter(
                    series.index, series.values, color=style["color"],
                    marker=style["marker"], s=8 if source == "obs" else 6,
                    alpha=0.52 if source == "obs" else 0.30,
                    edgecolors="none", zorder=4 if source == "obs" else 2,
                    rasterized=True,
                )
        ax.text(
            0.97, 0.91, station.upper(), transform=ax.transAxes,
            ha="right", va="top", fontsize=11, weight="bold",
            bbox=dict(facecolor="white", alpha=0.72, edgecolor="none", pad=1.0),
        )
        ax.set_xlim(START_TIME, END_TIME)
        ax.set_ylim(0, ymax)
        if metric == "direction":
            ax.set_yticks([0, 90, 180, 270, 360])
        ax.grid(axis="y", color="#D9D9D9", linewidth=0.45, alpha=0.55)

    for ax in axes_flat[len(station_order):]:
        ax.set_visible(False)

    ticks, labels = time_ticks(START_TIME, END_TIME)
    for col in range(ncols):
        active_rows = [row for row in range(nrows) if row * ncols + col < len(station_order)]
        if not active_rows:
            continue
        bottom = max(active_rows)
        for row in active_rows:
            ax = axes[row, col]
            ax.tick_params(axis="y", labelleft=(col == 0))
            ax.tick_params(axis="x", labelbottom=(row == bottom))
        axes[bottom, col].xaxis.set_major_locator(mticker.FixedLocator(ticks))
        axes[bottom, col].set_xticklabels(labels)

    handles = [
        Line2D(
            [0], [0], color=source_style(source)["color"],
            marker=source_style(source)["marker"],
            linestyle="-" if metric == "speed" else "None",
            linewidth=1.3, markersize=5.5, markeredgecolor="black",
            markeredgewidth=0.35, label=source_style(source)["label"],
        )
        for source in SOURCE_ORDER
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.955),
        ncol=len(handles),
        frameon=False,
        fontsize=14,
        handlelength=2.0,
        columnspacing=1.2,
    )
    fig.text(0.5, 0.026, f"Time (UTC, {START_TIME.year})", ha="center")
    ylabel = "Wind speed (m s$^{-1}$)" if metric == "speed" else "Wind direction (deg)"
    fig.text(0.035, 0.5, ylabel, ha="center", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.05, 0.045, 0.995, 0.925), w_pad=0.25, h_pad=0.18)
    output_path = PUBLISH_DIR / output_name
    save_tiff(fig, output_path)
    plt.show()
    plt.close(fig)
    print(f"Saved: {output_path}")
    return output_path


out_speed: dict[str, Path] = {}
out_direction: dict[str, Path] = {}
for averaging in AVERAGING_INTERVALS:
    speed_sources = all_station_sources(
        obs_speed_by_interval[averaging], simulation_speed_by_interval[averaging]
    )
    direction_sources = all_station_sources(
        obs_direction_by_interval[averaging], simulation_direction_by_interval[averaging]
    )
    out_speed[averaging] = plot_all_station_time_series(
        speed_sources, "speed", averaging,
        f"yangjiang_allstations_{averaging}_mean_wspd_{WINDOW_TAG}.tif",
    )
    out_direction[averaging] = plot_all_station_time_series(
        direction_sources, "direction", averaging,
        f"yangjiang_allstations_{averaging}_mean_wdir_{WINDOW_TAG}.tif",
    )


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_allstations_2min_mean_wspd_20250923-09_to_20250925-09.tif
Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_allstations_2min_mean_wdir_20250923-09_to_20250925-09.tif
Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_allstations_10min_mean_wspd_20250923-09_to_20250925-09.tif
Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_allstations_10min_mean_wdir_20250923-09_to_20250925-09.tif


### 3. Station-wise 2-min/10-min VRMSE, separate boxplots, and paper-ready statistics

VRMSE is calculated independently for the two averaging intervals. The station-level CSV contains one row per averaging interval, driver, and station; the summary CSV contains descriptive statistics for all ten interval-driver combinations. The two averaging intervals are exported as separate publication-style boxplots with shortened driver labels.


In [4]:
def wind_to_uv(speed: pd.DataFrame, direction: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    radians = np.deg2rad(direction)
    return -speed * np.sin(radians), -speed * np.cos(radians)


vrmse_records: list[dict] = []
for averaging in AVERAGING_INTERVALS:
    obs_u, obs_v = wind_to_uv(
        obs_speed_by_interval[averaging], obs_direction_by_interval[averaging]
    )
    for model in MODELS:
        sim_u, sim_v = wind_to_uv(
            simulation_speed_by_interval[averaging][model],
            simulation_direction_by_interval[averaging][model],
        )
        for station in station_order:
            aligned = pd.concat(
                [obs_u[station], obs_v[station], sim_u[station], sim_v[station]], axis=1,
                keys=["obs_u", "obs_v", "sim_u", "sim_v"],
            ).dropna()
            squared_vector_error = (
                (aligned["sim_u"] - aligned["obs_u"]) ** 2
                + (aligned["sim_v"] - aligned["obs_v"]) ** 2
            )
            vrmse_records.append({
                "averaging": averaging,
                "model": model,
                "station": station,
                "n_valid": int(len(aligned)),
                "vrmse": float(np.sqrt(squared_vector_error.mean())) if len(aligned) else np.nan,
            })

vrmse_long = pd.DataFrame(vrmse_records).merge(
    station_meta[["station", "lat", "lon", "elevation"]],
    on="station", how="left", validate="many_to_one",
)
if vrmse_long["vrmse"].isna().any() or not np.isfinite(vrmse_long["vrmse"]).all():
    raise RuntimeError("At least one averaging/model/station VRMSE is not finite")

summary_index = pd.MultiIndex.from_product(
    [AVERAGING_INTERVALS, MODELS], names=["averaging", "model"]
)
vrmse_summary = (
    vrmse_long.groupby(["averaging", "model"])["vrmse"]
    .agg(
        count="count",
        min="min",
        q1=lambda x: x.quantile(0.25),
        median="median",
        mean="mean",
        std="std",
        q3=lambda x: x.quantile(0.75),
        max="max",
    )
    .reindex(summary_index)
)

interval_label_map = {key: value["label"] for key, value in AVERAGING_STYLES.items()}
model_label_map = {key: value["label"] for key, value in MODEL_STYLES.items()}
vrmse_export = vrmse_long.assign(
    averaging=vrmse_long["averaging"].map(interval_label_map),
    model=vrmse_long["model"].map(model_label_map),
)
summary_export = vrmse_summary.reset_index()
summary_export["averaging"] = summary_export["averaging"].map(interval_label_map)
summary_export["model"] = summary_export["model"].map(model_label_map)

station_csv = PUBLISH_DIR / f"yangjiang_vrmse_station_{WINDOW_TAG}.csv"
summary_csv = PUBLISH_DIR / f"yangjiang_vrmse_summary_{WINDOW_TAG}.csv"
vrmse_export.to_csv(station_csv, index=False, float_format="%.6f")
summary_export.to_csv(summary_csv, index=False, float_format="%.6f")

print("VRMSE summary across 32 stations (m s^-1):")
display(summary_export.set_index(["averaging", "model"]).round(3))
print(f"Saved: {station_csv}")
print(f"Saved: {summary_csv}")


VRMSE summary across 32 stations (m s^-1):
                         count    min     q1  ...    std     q3     max
averaging model                               ...                      
2-min     Pangu-WRF         32  3.709  6.519  ...  3.089  9.275  20.413
          GraphCast-WRF     32  4.564  7.221  ...  3.060  9.685  20.748
          FengWu-WRF        32  4.156  7.294  ...  3.070  9.588  20.697
          FuXi-WRF          32  4.116  7.081  ...  3.060  9.597  20.754
          Aurora-WRF        32  4.273  7.341  ...  3.108  9.779  21.129
10-min    Pangu-WRF         32  3.908  6.408  ...  3.089  9.218  20.271
          GraphCast-WRF     32  4.780  7.162  ...  3.047  9.610  20.576
          FengWu-WRF        32  4.359  7.198  ...  3.030  9.533  20.484
          FuXi-WRF          32  4.253  6.933  ...  3.053  9.482  20.526
          Aurora-WRF        32  4.528  7.289  ...  3.065  9.677  20.803

[10 rows x 8 columns]
Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\

In [5]:
SHORT_MODEL_LABELS = {
    "pangu": "Pangu",
    "graphcast": "GraphCast",
    "fengwu": "FengWu",
    "fuxi": "FuXi",
    "aurora": "Aurora",
}


def plot_interval_vrmse_box(averaging: str) -> Path:
    fig, ax = plt.subplots(figsize=(10, 4.0))
    positions = np.arange(len(MODELS)) * 0.82
    data = [
        vrmse_long.loc[
            vrmse_long["model"].eq(model)
            & vrmse_long["averaging"].eq(averaging),
            "vrmse",
        ].dropna().to_numpy()
        for model in MODELS
    ]
    boxplot = ax.boxplot(
        data,
        positions=positions,
        widths=0.28,
        patch_artist=True,
        showmeans=True,
        meanline=False,
        showfliers=True,
        boxprops=dict(linewidth=1.3),
        medianprops=dict(linewidth=1.6),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        meanprops=dict(marker="o", markersize=5.5, markeredgewidth=1.0),
        flierprops=dict(
            marker="o", markersize=3.2, markerfacecolor="none",
            markeredgewidth=0.9, alpha=0.70,
        ),
    )

    # Match each model's boxplot color to the all-station time-series figures.
    for index, (model, box) in enumerate(zip(MODELS, boxplot["boxes"])):
        color = MODEL_STYLES[model]["color"]
        box.set_facecolor(mcolors.to_rgba(color, alpha=0.20))
        box.set_edgecolor(color)
        for artist in boxplot["whiskers"][2 * index:2 * index + 2]:
            artist.set_color(color)
        for artist in boxplot["caps"][2 * index:2 * index + 2]:
            artist.set_color(color)
        boxplot["medians"][index].set_color(color)
        boxplot["means"][index].set_markerfacecolor(color)
        boxplot["means"][index].set_markeredgecolor(color)
        boxplot["fliers"][index].set_markeredgecolor(color)

    ax.set_xticks(positions)
    ax.set_xticklabels([SHORT_MODEL_LABELS[model] for model in MODELS])
    ax.set_xlabel("Driver")
    ax.set_ylabel("VRMSE (m s$^{-1}$)")
    ax.set_ylim(bottom=0)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", linewidth=0.8, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()

    output_path = PUBLISH_DIR / f"yangjiang_vrmse_box_{averaging}_{WINDOW_TAG}.tif"
    save_tiff(fig, output_path)
    plt.show()
    plt.close(fig)
    print(f"Saved: {output_path}")
    return output_path


out_vrmse_box = {
    averaging: plot_interval_vrmse_box(averaging)
    for averaging in AVERAGING_INTERVALS
}

print("\nPaper-ready conclusions:")
for averaging in AVERAGING_INTERVALS:
    interval_summary = vrmse_summary.xs(averaging, level="averaging")
    median_ranking = interval_summary.sort_values("median")
    mean_ranking = interval_summary.sort_values("mean")
    average_label = AVERAGING_STYLES[averaging]["label"]
    print(
        f"- {average_label}: lowest median = {MODEL_STYLES[median_ranking.index[0]]['label']} "
        f"({median_ranking.iloc[0]['median']:.3f} m s^-1); lowest mean = "
        f"{MODEL_STYLES[mean_ranking.index[0]]['label']} "
        f"({mean_ranking.iloc[0]['mean']:.3f} m s^-1)."
    )
    print(
        f"  Median ranking: "
        + " < ".join(MODEL_STYLES[model]["label"] for model in median_ranking.index)
        + "."
    )

print("- Highest-VRMSE station by averaging interval and model:")
for averaging in AVERAGING_INTERVALS:
    for model in MODELS:
        row = (
            vrmse_long.loc[
                vrmse_long["averaging"].eq(averaging) & vrmse_long["model"].eq(model)
            ]
            .nlargest(1, "vrmse")
            .iloc[0]
        )
        print(
            f"  {AVERAGING_STYLES[averaging]['label']} {MODEL_STYLES[model]['label']}: "
            f"{row['station'].upper()} ({row['vrmse']:.3f}, n={int(row['n_valid'])})"
        )


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_vrmse_box_2min_20250923-09_to_20250925-09.tif
Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_vrmse_box_10min_20250923-09_to_20250925-09.tif

Paper-ready conclusions:
- 2-min: lowest median = Pangu-WRF (7.427 m s^-1); lowest mean = Pangu-WRF (8.136 m s^-1).
  Median ranking: Pangu-WRF < FuXi-WRF < Aurora-WRF < FengWu-WRF < GraphCast-WRF.
- 10-min: lowest median = Pangu-WRF (7.234 m s^-1); lowest mean = Pangu-WRF (8.057 m s^-1).
  Median ranking: Pangu-WRF < FuXi-WRF < FengWu-WRF < Aurora-WRF < GraphCast-WRF.
- Highest-VRMSE station by averaging interval and model:
  2-min Pangu-WRF: G2329 (20.413, n=49)
  2-min GraphCast-WRF: G2329 (20.748, n=49)
  2-min FengWu-WRF: G2329 (20.697, n=49)
  2-min FuXi-WRF: G2329 (20.754, n=49)
  2-min Aurora-WRF: G2329 (21.129, n=49)
  10-min Pangu-WRF: G2329 (20.271, n=49)
  10-min GraphCast-WRF: G2329 (20.576, n=49)
  10-min FengWu-WRF: G2329

### 4. Station-location distribution

Following the visual format of `Fig1/05_weather_stations.ipynb`, this map uses a clean 10 m Natural Earth land/ocean base, uniform station markers, direct station-code labels, dashed geographic gridlines, and manual label offsets with short leader lines in crowded areas.


In [6]:
def read_map_data(path: Path) -> dict[str, np.ndarray]:
    with nc.Dataset(path, "r") as dataset:
        return {name: np.asarray(dataset.variables[name][:]) for name in ["LON", "LAT", "HGT", "LU_INDEX"]}


map_data = read_map_data(MAP_FILE)
lon = map_data["LON"]
lat = map_data["LAT"]
terrain = map_data["HGT"].astype(float)
landuse = map_data["LU_INDEX"].astype(int)
water_mask = np.isin(landuse, [17, 21])
terrain_land = np.ma.masked_where(water_mask | ~np.isfinite(terrain) | (terrain < 0), terrain)

projection = ccrs.PlateCarree()
lon_pad = 0.025
lat_pad = 0.025
station_extent = [
    float(station_meta["lon"].min()) - lon_pad,
    float(station_meta["lon"].max()) + lon_pad,
    float(station_meta["lat"].min()) - lat_pad,
    float(station_meta["lat"].max()) + lat_pad,
]

fig = plt.figure(figsize=(6, 6))
ax = plt.axes(projection=projection)
ax.set_extent(station_extent, crs=projection)
ax.add_feature(cfeature.LAND.with_scale("10m"))
ax.add_feature(cfeature.OCEAN.with_scale("10m"))

grid = ax.gridlines(
    draw_labels=True,
    linestyle="--",
    linewidth=0.4,
    color="gray",
    alpha=0.6,
    crs=projection,
)
grid.top_labels = False
grid.right_labels = False
grid.xlocator = mticker.MultipleLocator(0.1)
grid.ylocator = mticker.MultipleLocator(0.1)
grid.xlabel_style = {"size": 12, "family": FONT_FAMILY}
grid.ylabel_style = {"size": 12, "family": FONT_FAMILY}

station_style = {
    "marker": "o",
    "s": 34,
    "facecolor": "#D1495B",
    "edgecolor": "white",
    "linewidth": 0.7,
}
ax.scatter(
    station_meta["lon"],
    station_meta["lat"],
    s=station_style["s"],
    marker=station_style["marker"],
    facecolors=station_style["facecolor"],
    edgecolors=station_style["edgecolor"],
    linewidths=station_style["linewidth"],
    transform=projection,
    zorder=4,
)

# Offsets are in points and are tuned for the dense north-central and
# south-western station clusters. Leader lines make displaced labels traceable.
label_offsets = {
    "gq040": (6, -5),
    "g2317": (6, -10),
    "g2328": (6, 4),
    "g2318": (6, 4),
    "gq013": (6, 3),
    "g2308": (-6, 4),
    "gq052": (-8, 12),
    "g2307": (-8, -10),
    "59663": (-6, -12),
    "g7312": (-10, 16),
    "gq020": (8, -9),
    "g2326": (8, 14),
    "gq055": (6, -10),
    "g2305": (6, -9),
    "g2304": (-5, 8),
    "g2329": (7, -8),
    "g7317": (6, -9),
    "gq008": (6, -9),
    "g2310": (6, -9),
    "g2320": (-6, 6),
    "g7311": (-6, 8),
}
leader_line_stations = {
    "gq040", "g2317", "gq052", "g2307", "59663", "g7312",
    "gq020", "g2326", "g2304", "g2329",
}

for row in station_meta.itertuples():
    offset_x, offset_y = label_offsets.get(row.station, (6, 3))
    ax.annotate(
        row.station.upper(),
        xy=(row.lon, row.lat),
        xytext=(offset_x, offset_y),
        textcoords="offset points",
        ha="left" if offset_x >= 0 else "right",
        va="bottom" if offset_y >= 0 else "top",
        fontsize=8.5,
        fontfamily=FONT_FAMILY,
        arrowprops={"arrowstyle": "-", "color": "0.45", "lw": 0.35}
        if row.station in leader_line_stations else None,
        transform=projection,
        zorder=6,
    )

legend_handle = Line2D(
    [0], [0],
    marker=station_style["marker"],
    linestyle="None",
    markerfacecolor=station_style["facecolor"],
    markeredgecolor=station_style["edgecolor"],
    markeredgewidth=station_style["linewidth"],
    markersize=7,
    label="Weather station",
)
ax.legend(
    handles=[legend_handle],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    frameon=False,
    prop={"family": FONT_FAMILY, "size": 10},
    handletextpad=0.6,
)

out_station_map = PUBLISH_DIR / "yangjiang_station_locations.tif"
save_tiff(fig, out_station_map)
plt.show()
plt.close(fig)
print(f"Saved: {out_station_map}")


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_station_locations.tif


### 5. Yangjiang terrain and land-use/LCZ maps

Terrain is masked over WRF water categories and displayed with hillshade. The discrete land-use/LCZ map uses only classes present in the NetCDF file, including the urban LCZ extension (32–40).


In [7]:
LU_ALL = {
    1: ("#05450A", "Evergreen needleleaf forest"),
    2: ("#086A10", "Evergreen broadleaf forest"),
    3: ("#54A708", "Deciduous needleleaf forest"),
    4: ("#78D203", "Deciduous broadleaf forest"),
    5: ("#009900", "Mixed forests"),
    6: ("#C6B044", "Closed shrublands"),
    7: ("#DCD159", "Open shrublands"),
    8: ("#DADE48", "Woody savannas"),
    9: ("#FBFF13", "Savannas"),
    10: ("#B6FF05", "Grasslands"),
    11: ("#27FF87", "Permanent wetlands"),
    12: ("#C24F44", "Croplands"),
    13: ("#A5A5A5", "Urban and built-up"),
    14: ("#FF6D4C", "Cropland/natural vegetation mosaic"),
    15: ("#69FFF8", "Snow and ice"),
    16: ("#F9FFA4", "Barren or sparsely vegetated"),
    17: ("#1C0DFF", "Water"),
    18: ("#6E8B3D", "Wooded tundra"),
    19: ("#9ACD32", "Mixed tundra"),
    20: ("#D2B48C", "Barren tundra"),
    21: ("#4169E1", "Lakes"),
    31: ("#910613", "LCZ 1 compact high-rise"),
    32: ("#D9081C", "LCZ 2 compact mid-rise"),
    33: ("#FF0A22", "LCZ 3 compact low-rise"),
    34: ("#C54F1E", "LCZ 4 open high-rise"),
    35: ("#FF6628", "LCZ 5 open mid-rise"),
    36: ("#FF985E", "LCZ 6 open low-rise"),
    37: ("#FDED3F", "LCZ 7 lightweight low-rise"),
    38: ("#BBBBBB", "LCZ 8 large low-rise"),
    39: ("#FFCBAB", "LCZ 9 sparsely built"),
    40: ("#565656", "LCZ 10 heavy industry"),
}

terrain_colors = [
    (0.00, "#FFFFFF"), (0.04, "#D8F0C8"), (0.12, "#A8D890"),
    (0.25, "#78B850"), (0.38, "#C8C060"), (0.52, "#C8A050"),
    (0.65, "#A07038"), (0.80, "#784820"), (1.00, "#4A2810"),
]
terrain_cmap = mcolors.LinearSegmentedColormap.from_list("yangjiang_terrain", terrain_colors, N=512)

MAP_FONTSIZE = 16
MAP_FIGSIZE = (10, 8)
MAP_GRID_STEP = 0.1


def add_gridlines(ax, *, fontsize: int = MAP_FONTSIZE):
    grid = ax.gridlines(
        draw_labels=True, linewidth=0.4, color="gray", alpha=0.5,
        linestyle="--", crs=projection,
    )
    grid.top_labels = False
    grid.right_labels = False
    grid.xlocator = mticker.MultipleLocator(MAP_GRID_STEP)
    grid.ylocator = mticker.MultipleLocator(MAP_GRID_STEP)
    grid.xlabel_style = {"size": fontsize, "family": FONT_FAMILY}
    grid.ylabel_style = {"size": fontsize, "family": FONT_FAMILY}
    return grid


def add_aligned_colorbar(
    fig, ax, mappable, *, label=None, tick_values=None, tick_labels=None,
    fontsize: int = MAP_FONTSIZE, width: float = 0.020, pad: float = 0.015,
):
    fig.canvas.draw()
    ax_position = ax.get_position()
    colorbar_ax = fig.add_axes(
        [ax_position.x1 + pad, ax_position.y0, width, ax_position.height]
    )
    colorbar = fig.colorbar(mappable, cax=colorbar_ax, orientation="vertical")
    if label is not None:
        colorbar.set_label(label, fontsize=fontsize, family=FONT_FAMILY)
    if tick_values is not None:
        colorbar.set_ticks(tick_values)
    if tick_labels is not None:
        colorbar.set_ticklabels(tick_labels, fontsize=fontsize - 1)
    colorbar.ax.tick_params(labelsize=fontsize)
    return colorbar


def save_map_tiff(fig, path: Path, map_ax, colorbar, gridliner) -> Path:
    """Save Cartopy maps with a finite bbox that includes map, labels, and colorbar."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    candidate_bboxes = [
        map_ax.get_window_extent(renderer),
        colorbar.ax.get_tightbbox(renderer),
        *[artist.get_window_extent(renderer) for artist in gridliner.label_artists],
    ]
    finite_bboxes = [
        bbox for bbox in candidate_bboxes
        if bbox is not None and np.isfinite(np.asarray(bbox.bounds, dtype=float)).all()
    ]
    bbox_pixels = mpl.transforms.Bbox.union(finite_bboxes)
    bbox_inches = bbox_pixels.transformed(fig.dpi_scale_trans.inverted()).padded(0.06)
    kwargs = dict(dpi=DPI, format="tiff", bbox_inches=bbox_inches)
    try:
        fig.savefig(path, pil_kwargs={"compression": "tiff_lzw"}, **kwargs)
    except TypeError:
        fig.savefig(path, **kwargs)
    return path


domain_extent = [
    float(np.nanmin(lon)), float(np.nanmax(lon)),
    float(np.nanmin(lat)), float(np.nanmax(lat)),
]


In [8]:
valid_terrain = terrain_land.compressed()
terrain_min = float(np.nanmin(valid_terrain))
terrain_max = float(np.nanmax(valid_terrain))
terrain_tick_step = max(1, round((terrain_max - terrain_min) / 6 / 50) * 50)
terrain_tick_step = 50 if terrain_tick_step == 0 else terrain_tick_step
terrain_vmin = math.floor(terrain_min / terrain_tick_step) * terrain_tick_step
terrain_vmax = math.ceil(terrain_max / terrain_tick_step) * terrain_tick_step
terrain_norm = mcolors.Normalize(vmin=terrain_vmin, vmax=terrain_vmax)

lat_mid = float((domain_extent[2] + domain_extent[3]) / 2)
dx_deg = float(np.nanmean(np.abs(np.diff(lon, axis=1))))
dy_deg = float(np.nanmean(np.abs(np.diff(lat, axis=0))))
dx_m = dx_deg * 111320 * math.cos(math.radians(lat_mid))
dy_m = dy_deg * 110570
light_source = LightSource(azdeg=315, altdeg=45)
terrain_for_shade = np.where(terrain_land.mask, 0.0, terrain_land.filled(0.0))
hillshade = light_source.hillshade(
    terrain_for_shade, vert_exag=3, dx=dx_m, dy=dy_m
)
hillshade = np.where(terrain_land.mask, np.nan, hillshade)
terrain_rgb = terrain_cmap(terrain_norm(terrain_land.filled(np.nan)))
terrain_blend = light_source.blend_hsv(
    terrain_rgb[:, :, :3], hillshade[:, :, np.newaxis],
    hsv_max_sat=0.7, hsv_max_val=0.9,
)
terrain_rgba = np.dstack(
    [terrain_blend, np.where(terrain_land.mask, 0.0, 1.0)]
)

fig = plt.figure(figsize=MAP_FIGSIZE)
ax = fig.add_subplot(1, 1, 1, projection=projection)
ax.set_extent(domain_extent, crs=projection)
ax.set_facecolor("white")
origin = "lower" if float(np.nanmean(lat[-1])) > float(np.nanmean(lat[0])) else "upper"
ax.imshow(
    terrain_rgba, origin=origin, extent=domain_extent,
    transform=projection, interpolation="bilinear", zorder=1,
)
terrain_gridliner = add_gridlines(ax)
terrain_scalar = mpl.cm.ScalarMappable(cmap=terrain_cmap, norm=terrain_norm)
terrain_scalar.set_array(np.array([terrain_vmin, terrain_vmax], dtype=np.float32))
terrain_ticks = np.arange(
    terrain_vmin, terrain_vmax + 0.1 * terrain_tick_step, terrain_tick_step
)
terrain_colorbar = add_aligned_colorbar(
    fig, ax, terrain_scalar, label="Elevation (m)",
    tick_values=terrain_ticks,
    tick_labels=[f"{int(value)}" for value in terrain_ticks],
)
out_terrain = PUBLISH_DIR / "yangjiang_hgt_wrfgrid.tif"
save_map_tiff(fig, out_terrain, ax, terrain_colorbar, terrain_gridliner)
plt.show()
plt.close(fig)
print(
    f"Saved: {out_terrain} (elevation {terrain_min:.1f}-{terrain_max:.1f} m)"
)


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_hgt_wrfgrid.tif


In [9]:
present_classes = [int(value) for value in sorted(np.unique(landuse)) if int(value) in LU_ALL]
colors = [LU_ALL[value][0] for value in present_classes]
labels = [LU_ALL[value][1] for value in present_classes]
class_to_plot_index = {value: index for index, value in enumerate(present_classes)}
landuse_plot = np.full(landuse.shape, np.nan, dtype=float)
for value, plot_index in class_to_plot_index.items():
    landuse_plot[landuse == value] = plot_index
boundaries = np.arange(-0.5, len(present_classes) + 0.5, 1.0)
lu_cmap = ListedColormap(colors)
lu_norm = BoundaryNorm(boundaries, ncolors=len(colors))

fig = plt.figure(figsize=MAP_FIGSIZE)
ax = fig.add_subplot(1, 1, 1, projection=projection)
ax.set_extent(domain_extent, crs=projection)
ax.set_facecolor("white")
image = ax.pcolormesh(
    lon, lat, landuse_plot, cmap=lu_cmap, norm=lu_norm,
    transform=projection, shading="auto", rasterized=True,
)
lcz_gridliner = add_gridlines(ax)
lcz_colorbar = add_aligned_colorbar(
    fig, ax, image, tick_values=np.arange(len(present_classes)), tick_labels=labels,
)
out_lcz = PUBLISH_DIR / "yangjiang_lcz_wrfgrid.tif"
save_map_tiff(fig, out_lcz, ax, lcz_colorbar, lcz_gridliner)
plt.show()
plt.close(fig)
print(f"Saved: {out_lcz}")


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Supplementary\publish\yangjiang_lcz_wrfgrid.tif


## Takeaways

- For 2-min averages, Pangu-WRF has the lowest station median VRMSE (7.427 m s⁻¹) and mean VRMSE (8.136 m s⁻¹).
- For 10-min averages, Pangu-WRF also has the lowest median VRMSE (7.234 m s⁻¹) and mean VRMSE (8.057 m s⁻¹).
- The 2-min median ranking is Pangu-WRF < FuXi-WRF < Aurora-WRF < FengWu-WRF < GraphCast-WRF; the 10-min ranking is Pangu-WRF < FuXi-WRF < FengWu-WRF < Aurora-WRF < GraphCast-WRF.
- The 10-min median and mean VRMSE are slightly lower than the matching 2-min values for all five drivers.
- G2329 is the largest-error station for all ten averaging/model combinations (20.271–21.129 m s⁻¹).
- The station CSV contains 320 rows: 2 averaging intervals × 5 drivers × 32 stations. Valid hourly-pair counts range from 42 to 49 and remain attached to every station result.
- These results describe association and forecast error; they do not by themselves establish a causal terrain or LCZ effect.


In [10]:
expected_outputs = [
    *out_speed.values(),
    *out_direction.values(),
    *out_vrmse_box.values(),
    station_csv,
    summary_csv,
    out_station_map,
    out_terrain,
    out_lcz,
]
missing_outputs = [path for path in expected_outputs if not path.is_file() or path.stat().st_size == 0]
if missing_outputs:
    raise RuntimeError(f"Missing or empty outputs: {missing_outputs}")

print("Output check passed:")
for path in expected_outputs:
    print(f"- {path.name}: {path.stat().st_size / 1024:.1f} KiB")


Output check passed:
- yangjiang_allstations_2min_mean_wspd_20250923-09_to_20250925-09.tif: 10530.9 KiB
- yangjiang_allstations_10min_mean_wspd_20250923-09_to_20250925-09.tif: 11134.2 KiB
- yangjiang_allstations_2min_mean_wdir_20250923-09_to_20250925-09.tif: 8477.3 KiB
- yangjiang_allstations_10min_mean_wdir_20250923-09_to_20250925-09.tif: 8471.8 KiB
- yangjiang_vrmse_box_2min_20250923-09_to_20250925-09.tif: 671.1 KiB
- yangjiang_vrmse_box_10min_20250923-09_to_20250925-09.tif: 670.7 KiB
- yangjiang_vrmse_station_20250923-09_to_20250925-09.csv: 21.1 KiB
- yangjiang_vrmse_summary_20250923-09_to_20250925-09.csv: 0.9 KiB
- yangjiang_station_locations.tif: 1120.7 KiB
- yangjiang_hgt_wrfgrid.tif: 16109.2 KiB
- yangjiang_lcz_wrfgrid.tif: 4009.6 KiB
